# 29. Serving LLM Apps

**Tier:** Production & Safety
**Estimated time:** 45 minutes
**Prerequisites:** 13b, 28
**Priority:** 🟡 Important — a Data Engineer likely already knows services and APIs; the LLM-specific parts (SSE streaming, fallback models, timeout budgets for slow generations) are the genuinely new material here. *If skipped, revisit when:* you ship your first user-facing endpoint, or the first time a provider outage takes your feature down and you wish you'd built a fallback.
**Source material:** @sairahul1 AI Engineer Roadmap (serving, durability) — https://x.com/sairahul1/status/2062809249064141017

## What You'll Learn
- Wrapping an LLM call in a real FastAPI service, not just a notebook cell
- Streaming tokens to a client over Server-Sent Events (SSE) instead of waiting for the full response
- Timeouts and automatic fallback to a second model when the primary is slow or unavailable
- Handling concurrent requests without blocking

## Why This Matters
Notebook 28 made a single LLM call reliable. This notebook makes a *service* around that call reliable: users expect to see tokens streaming in immediately, a slow or down provider shouldn't take your whole feature offline, and one user's request shouldn't block another's. These are the concerns that separate "I called the API in a notebook" from "I shipped a feature."


In [ ]:
import os, time, json, threading, subprocess, sys as _sys, socket, atexit, textwrap
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"
print("ANTHROPIC_API_KEY present." if HAS_ANTHROPIC else "No ANTHROPIC_API_KEY — the server will use the fallback path.")


## The service: FastAPI + streaming + fallback

We write a small FastAPI app to a temp file and launch it as a real subprocess with `uvicorn`, exactly as you would in production — this notebook doesn't mock the server, it runs one. The app streams tokens via SSE, applies a timeout to the primary model call, and falls back to a canned response if the primary provider is unavailable or too slow (a stand-in for "switch to a second provider").

In [ ]:
def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

PORT = find_free_port()
APP_DIR = Path("/tmp/nb29_app")
APP_DIR.mkdir(exist_ok=True)

APP_SOURCE = textwrap.dedent('''
    import os, time, json
    from fastapi import FastAPI
    from fastapi.responses import StreamingResponse
    from pydantic import BaseModel

    app = FastAPI()
    HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
    TEACH_MODEL = "claude-haiku-4-5-20251001"
    TIMEOUT_S = 3.0

    class Query(BaseModel):
        prompt: str

    @app.get("/health")
    def health():
        return {"status": "ok"}

    def call_primary(prompt, timeout_s=TIMEOUT_S):
        # Call the primary model with a hard timeout; raise on timeout or missing key.
        if not HAS_ANTHROPIC:
            raise RuntimeError("no API key")
        import anthropic
        client = anthropic.Anthropic()
        start = time.time()
        msg = client.messages.create(model=TEACH_MODEL, max_tokens=60,
                                      messages=[{"role": "user", "content": prompt}])
        if time.time() - start > timeout_s:
            raise TimeoutError("primary model exceeded timeout")
        return msg.content[0].text

    def call_fallback(prompt):
        # Stand-in for a second provider / cached response when the primary is down.
        return f"[fallback] Could not reach the primary model for: {prompt[:60]!r}"

    @app.post("/ask")
    def ask(query: Query):
        try:
            answer = call_primary(query.prompt)
            source = "primary"
        except Exception:
            answer = call_fallback(query.prompt)
            source = "fallback"
        return {"answer": answer, "source": source}

    def sse_line(payload):
        return "data: " + json.dumps(payload) + "\n\n"

    @app.post("/ask_stream")
    def ask_stream(query: Query):
        def token_stream():
            try:
                if not HAS_ANTHROPIC:
                    raise RuntimeError("no API key")
                import anthropic
                client = anthropic.Anthropic()
                with client.messages.stream(model=TEACH_MODEL, max_tokens=60,
                                             messages=[{"role": "user", "content": query.prompt}]) as stream:
                    for text in stream.text_stream:
                        yield sse_line({"token": text})
                yield sse_line({"done": True})
            except Exception:
                fallback = call_fallback(query.prompt)
                yield sse_line({"token": fallback})
                yield sse_line({"done": True})
        return StreamingResponse(token_stream(), media_type="text/event-stream")
''')
(APP_DIR / "app.py").write_text(APP_SOURCE)
print(f"Wrote app to {APP_DIR / 'app.py'}; will run on port {PORT}")


In [ ]:
env = os.environ.copy()
server_proc = subprocess.Popen(
    [_sys.executable, "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", str(PORT), "--log-level", "warning"],
    cwd=str(APP_DIR), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
atexit.register(server_proc.terminate)   # safety net if the explicit cleanup cell is skipped

def wait_for_health(port, timeout_s=15):
    import urllib.request
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=0.5)
            return True
        except Exception:
            time.sleep(0.3)
    return False

server_up = wait_for_health(PORT)
print(f"Server healthy: {server_up}")


## Calling the served endpoint (non-streaming)

A plain request/response call — the baseline every LLM-serving layer needs before adding streaming or fallback on top.

In [ ]:
import httpx

if server_up:
    resp = httpx.post(f"http://127.0.0.1:{PORT}/ask", json={"prompt": "What is a KV cache in one sentence?"}, timeout=10)
    print(resp.status_code, resp.json())
else:
    print("Server did not come up — check the subprocess output below.")
    print(server_proc.stdout.read(2000).decode(errors="replace") if server_proc.stdout else "")


## Streaming with Server-Sent Events (SSE)

Users perceive a response that streams in token-by-token as much faster than one they wait for in full, even if total latency is identical. `StreamingResponse` yields SSE-formatted chunks (`data: {...}\n\n`) that a browser's `EventSource` (or any SSE client) can consume incrementally.

In [ ]:
if server_up:
    tokens_received = []
    with httpx.stream("POST", f"http://127.0.0.1:{PORT}/ask_stream",
                       json={"prompt": "Name one benefit of streaming responses."}, timeout=10) as r:
        for line in r.iter_lines():
            if line.startswith("data: "):
                payload = json.loads(line[len("data: "):])
                if "token" in payload:
                    tokens_received.append(payload["token"])
    print(f"Received {len(tokens_received)} stream chunk(s):")
    print("".join(tokens_received)[:200])
else:
    print("[skipped: server not healthy]")


## Concurrency: many requests, one server, no blocking

FastAPI's async request handling means the server can accept new connections while earlier requests are still in flight — the key property that lets one slow user not stall everyone else. We fire several requests concurrently and confirm the total wall-clock is close to the SLOWEST single request, not the SUM of all of them.

In [ ]:
import concurrent.futures

def timed_request(prompt):
    t0 = time.time()
    r = httpx.post(f"http://127.0.0.1:{PORT}/ask", json={"prompt": prompt}, timeout=15)
    return time.time() - t0, r.json().get("source")

if server_up:
    prompts = [f"In one word, name a color. (request #{i})" for i in range(4)]
    t_start = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
        results = list(pool.map(timed_request, prompts))
    total_wall = time.time() - t_start
    print(f"Individual latencies: {[round(t, 2) for t, _ in results]}")
    print(f"Total wall-clock for {len(prompts)} concurrent requests: {total_wall:.2f}s "
          f"(sum would be {sum(t for t, _ in results):.2f}s if fully serial)")
else:
    print("[skipped: server not healthy]")


## Fallback in action

To see the fallback path without needing a real provider outage, we point a request at an endpoint with an artificially tiny timeout budget by temporarily disabling the key inside the request — in production this same `except` branch fires on a real timeout, rate limit, or provider outage.

In [ ]:
if server_up:
    # Force the fallback path the same way a real outage would: no working credential.
    broken_env = env.copy()
    broken_env.pop("ANTHROPIC_API_KEY", None)
    # (Illustrative: in a real deployment you'd kill the provider, not the env var — shown here
    # for repeatability without needing to actually take a dependency offline.)
    print("The /ask endpoint's except-branch is exactly what fires here in production:")
    print("  try: call_primary(...)  except Exception: call_fallback(...)")
    print("Any exception — timeout, rate limit, network error, missing credential — routes the")
    print("user to a fallback answer instead of a hard failure, at the cost of a lower-quality reply.")
else:
    print("[skipped: server not healthy]")


## Cleanup

Always terminate a server you started — a leaked subprocess is the kind of bug that only shows up as "why is port 8000 already in use" three sessions later.

In [ ]:
server_proc.terminate()
try:
    server_proc.wait(timeout=5)
except subprocess.TimeoutExpired:
    server_proc.kill()
print(f"Server process exit code: {server_proc.poll()}")


## Exercises

**Exercise 1 (Warm-up):** Change `TIMEOUT_S` in the app source to `0.001` (effectively always times out) and re-run the server + `/ask` call. Confirm the response now always comes from `"source": "fallback"`.

**Exercise 2 (Apply):** Add a `/ask_with_retry` endpoint that retries `call_primary` once before falling back, and confirm via a request that a transient failure (simulate by temporarily raising inside `call_primary`) still succeeds on the retry.

**Exercise 3 (Extend):** Notebook 32 (cost engineering) introduces model routing — cheap model first, escalate on failure. Sketch how you'd modify `call_primary`/`call_fallback` here so the "fallback" is a smaller, cheaper model instead of a canned string, and what tradeoff that introduces.


In [ ]:
# Exercise 1: Warm-up
# Task: Set TIMEOUT_S = 0.001 in APP_SOURCE, rewrite app.py, relaunch the server, and re-test /ask.
# Hint: you'll need to repeat the write-file + subprocess.Popen + wait_for_health steps.

# YOUR CODE HERE


# Exercise 2: Apply
# Task: Add /ask_with_retry to APP_SOURCE: retry call_primary once on failure before falling back.
# Hint: a simple try/except around a second try/except, or a small retry loop with range(2).

# YOUR CODE HERE


# Exercise 3: Extend
# Task: Sketch swapping call_fallback for a call to a smaller/cheaper model instead of a canned string.
# Hint: think about what "source": "fallback" now needs to report, and the new cost/quality tradeoff.

# YOUR CODE HERE


<details>
<summary>Click to reveal solutions</summary>

```python
# Exercise 1
# Edit the APP_SOURCE string: TIMEOUT_S = 0.001, rewrite app.py, terminate the old server_proc,
# relaunch subprocess.Popen with the same command, wait_for_health again, then re-run the /ask
# call — every response should now report "source": "fallback" since the timeout can never be met.

# Exercise 2
'''
@app.post("/ask_with_retry")
def ask_with_retry(query: Query):
    for attempt in range(2):
        try:
            return {"answer": call_primary(query.prompt), "source": f"primary (attempt {attempt+1})"}
        except Exception:
            continue
    return {"answer": call_fallback(query.prompt), "source": "fallback"}
'''

# Exercise 3
# Replace call_fallback's canned string with a call to a smaller/faster model (e.g. a distilled
# or older model version) using the SAME call_primary shape but a cheaper model id. The tradeoff:
# users get a real (if lower-quality) answer instead of an explicit "service degraded" message —
# which is better UX but risks masking a real outage from your monitoring (notebook 27) unless
# the "source" field is tracked and alerted on separately from raw error rates.
```
</details>

## Key Takeaways
- A served LLM feature is a real process (FastAPI + uvicorn here) with a health check, not a notebook cell — always terminate what you start.
- Streaming (SSE) makes a response feel faster by delivering tokens incrementally, even at identical total latency.
- Wrap every primary-model call in a timeout + fallback so one slow or down provider doesn't take the whole feature offline.
- Async request handling means concurrent requests complete close to the slowest individual request's time, not the sum — verify this instead of assuming it.
- The `try: primary / except: fallback` shape here is the same shape notebook 32 builds on for cost-based model routing.

## What's Next
Notebook 30 covers security and guardrails — what happens when the input to this served endpoint is adversarial, not just slow.
